# 2025-07-26: Annotate the bone marrow data
### By [Aishwarya Chander](aishwarya.chander@alleninstitute.org), High Resolution Translational Immunology, Allen Institute for Immunology
**Main Aim**: Here, I look into key marker genes to identify cell types in the bone marrow data. Overclustered groups are identified by iterative dot plot anlaysis for noisy genes and leiden clustering in ran. Wherever necessary UMAP and TSNEs were generated independently to confirm labels and marker genes in a temporary console kernel.

## 1. Imports

In [1]:
import anndata
import scanpy as sc
import matplotlib.pyplot as plt
import os
import anndata as ad
import numpy as np
import pandas as pd

sc.settings.n_jobs = 30
sc.settings.verbosity = 0

plt.rcParams['figure.dpi'] = 80
plt.rcParams['figure.figsize'] = (5, 5)

## 2. Dendritic Cells Clean Up

In [ ]:
adata = sc.read_h5ad('../../../data/rna/tmp/bmmc-annotation/dcs-processed-v1.h5ad')

In [ ]:
# Optional: compute dendrogram for grouping
sc.tl.dendrogram(adata, groupby='leiden')

# Dotplot
sc.pl.dotplot(
    adata,
    var_names=[
        # Dendritic Cells
        'AXL', 'SIGLEC6', 'LILRA4',       # dc_asdc
        'CLEC9A', 'BATF3', 'XCR1',        # dc_cdc1
        'CD1C', 'FCER1A', 'CLEC10A',      # dc_cdc2
        'CD14',                           # dc_cdc2-cd14.pos
        'IFI6', 'ISG15', 'MX1',           # dc_cdc2-isg.pos
        'IL3RA', 'TCF4',                  # dc_pdc

        # Monocytes
        'CD14', 'LYZ', 'VCAN',            # mono_cd14
        'IFI6', 'ISG15', 'MX1',           # mono_cd14_isg_high

        # Progenitors
        'DNTT', 'CD7', 'IL7R',            # prog_clp
        'MPO', 'ELANE',                   # prog_cmp
        'IRF8', 'CSF1R',                  # prog_cmp_mono
        'IRF8', 'FLT3', 'ID2',            # prog_dc
        'CD1C', 'IRF4', 'FLT3',           # prog_dc_cdc
        'LILRA4', 'IRF7', 'IL3RA',        # prog_dc_pdc
        'CD34', 'SOX4', 'GATA2',          # prog_lmpp
        'HBB', 'HBA1', 'KLF1', 'AHSP'     # prog_mature_ery_polychromatic
    ],
    groupby='leiden',
    standard_scale='var',
    dendrogram=True,
    cmap='Reds',
    figsize=(20, 8),
    show=False
)

# Save as PNG
plt.savefig('figures/dc_dotplot_l3_markers.png', dpi=300, bbox_inches='tight')
plt.close()

In [ ]:
leiden_to_label = {'0': 'mono_cd14',
 '1': 'dc_pdc',
 '2': 'dc_pdc',
 '3': 'dc_cdc1',
 '4': 'prog_dc_cdc',
 '5': 'dc_asdc',
 '6': 'prog_dc_cdc',
 '7': 'prog_dc_cdc',
 '8': 'dc_cdc2',
 '9': 'prog_dc_pdc',
 '10': 'dc_cdc2',
 '11': 'prog_dc_pdc',
 '12': 'dc_cdc2',
 '13': 'dc_cdc2-isg.pos',
 '14': 'dc_pdc',
 '15': 'dc_pdc',
 '16': 'dc_cdc2',
 '17': 'prog_dc_pdc',
 '18': 'dc_cdc2',
 '19': 'mono_cd14',
 '20': 'dc_cdc2',
 '21': 'dc_cdc2',
 '22': 'dc_cdc1',
 '23': 'prog_dc_cdc'}

adata.obs['aifi_celltype_l3'] = adata.obs['leiden'].map(leiden_to_label).astype('category').cat.remove_unused_categories()

In [ ]:
adata.obs[['cell_uuid', 'aifi_celltype_l3']].to_parquet('../../../data/rna/bmmc-labels/dc_labels.parquet')

## 2. Monocytes Clean up

In [ ]:
adata = sc.read_h5ad('../../../data/rna/tmp/bmmc-annotation/monocytes-processed-v1.h5ad')

In [ ]:
## Subcluster groups
sc.tl.leiden(
    adata,
    restrict_to=('leiden', ['11']),
    resolution=0.3,
    n_iterations=5
)

In [ ]:
sc.tl.dendrogram(adata, groupby='leiden_R')

# Dotplot for monocyte subset markers
sc.pl.dotplot(
    adata,
    var_names=[
        # Classical monocytes
        'CD14', 'LYZ', 'VCAN',

        # Inflammatory / IFN-stimulated monocytes
        'IFI6', 'ISG15', 'MX1', 'GBP1', 'GBP2', 'GBP4', 'GBP5',

        # Non-classical/Intermediate monocytes & activation
        'FCGR3A', 'S100A9', 'S100A10', 'MKI67'
    ],
    groupby='leiden_R',
    standard_scale='var',
    dendrogram=True,
    cmap='Reds',
    figsize=(7, 7),
    show=False
)

plt.savefig('figures/monocyte_dotplot_subsets.png', dpi=300, bbox_inches='tight')
plt.close()

In [ ]:
leiden_to_label = {'0': 'mono_cd16',
 '1': 'mono_cd14',
 '2': 'mono_cd14',
 '3': 'mono_cd14',
 '4': 'mono_cd14',
 '5': 'mono_cd14',
 '6': 'mono_cd14_isg',
 '7': 'mono_cd14',
 '8': 'mono_cd14',
 '9': 'mono_cd14',
 '10': 'mono_cd14',
 '11,0': 'mono_intermediate',
 '11,1': 'mono_cd14',
 '12': 'mono_cd14_isg',
 '13': 'junk'}

adata.obs['aifi_celltype_l3'] = adata.obs['leiden_R'].map(leiden_to_label).astype('category').cat.remove_unused_categories()

In [ ]:
adata.obs[['cell_uuid', 'aifi_celltype_l3']].to_parquet('../../../data/rna/bmmc-labels/monocytes_labels.parquet')

## 3.  B Cells Clean up

In [ ]:
adata = sc.read_h5ad('../../../data/rna/tmp/bmmc-annotation/b_cells-processed-v1.h5ad')

In [ ]:
## Subcluster groups
sc.tl.leiden(
    adata,
    restrict_to=('leiden', ['12']),
    resolution=0.3,
    n_iterations=5
)

sc.tl.leiden(
    adata,
    restrict_to=('leiden_R', ['6']),
    resolution=0.3,
    n_iterations=5
)

In [ ]:
sc.tl.dendrogram(adata, groupby='leiden_R')  # replace with your B cell clustering key

sc.pl.dotplot(
    adata,
    var_names=[
        # Precursor B (Pro/Pre)
        'DNTT', 'RAG1', 'RAG2', 'VPREB1', 'CD34', 'EBF1', 'PAX5',

        # Core Naive B
        'CD19', 'CD79A', 'MS4A1', 'TCL1A', 'IL4R',

        # Transitional B
        'CD24', 'CD38', 'TCL1A', 'IL10RA',

        # Activated Naive B
        'CD69', 'CD83', 'CD86',

        # ISG⁺ B Cells
        'IFI6', 'ISG15', 'MX1', 'IFIT1', 'OAS1', 'RSAD2', 'BST2', 'STAT1',

        # Core Memory B
        'CD27', 'AIM2', 'BANK1', 'TNFRSF13B',

        # Effector Memory B
        'ITGAX', 'TBX21', 'ZEB2', 'PRDM1', 'CD52',

        # Activated Memory B
        'CD69', 'CD83', 'CD86', 'FOS', 'JUN',

        # CD95 Memory
        'FAS', 'CD27', 'FCRL5',
        
        # Contamination
        'HBB', 'HBA1', 'AHSP'
    ],
    groupby='leiden_R',  # update to your relevant cluster key
    standard_scale='var',
    dendrogram=True,
    cmap='Reds',
    figsize=(20, 6),
    show=False
)

plt.savefig('figures/bcell_dotplot.png', dpi=300, bbox_inches='tight')
plt.close()

In [ ]:
leiden_to_label = {'0': 'b_precursor_proliferating',
 '1': 'b_precursor_proliferating',
 '2': 'b_precursor_proliferating',
 '3': 'b_precursor_lcr',
 '4': 'b_naive',
 '5': 'b_transitional',
 '6,0': 'b_transitional',
 '6,1': 'b_transitional',
 '6,2': 'b_transitional_isg',
 '7': 'b_precursor_lcr',
 '8': 'b_precursor_proliferating',
 '9': 'b_precursor_isg',
 '10': 'b_naive',
 '11': 'b_naive',
 '12,0': 'b_memory_core',
 '12,1': 'b_memory_core',
 '12,2': 'b_memory_cd95',
 '13': 'b_precursor_hcr',
 '14': 'b_naive_isg',
 '15': 'junk'}

adata.obs['aifi_celltype_l3'] = adata.obs['leiden_R'].map(leiden_to_label).astype('category').cat.remove_unused_categories()

In [ ]:
adata.obs[['cell_uuid', 'aifi_celltype_l3']].to_parquet('../../../data/rna/bmmc-labels/b_cells_labels.parquet')

## 4. Progenitor Cells Clean up

In [ ]:
adata = sc.read_h5ad('../../../data/rna/tmp/bmmc-annotation/progenitors-processed-v1.h5ad')

In [ ]:
## Subcluster groups
sc.tl.leiden(
    adata,
    restrict_to=('leiden', ['4']),
    resolution=0.3,
    n_iterations=5
)

sc.tl.leiden(
    adata,
    restrict_to=('leiden_R', ['10']),
    resolution=0.3,
    n_iterations=5
)

sc.tl.leiden(
    adata,
    restrict_to=('leiden_R', ['11']),
    resolution=0.3,
    n_iterations=5
)

sc.tl.leiden(
    adata,
    restrict_to=('leiden_R', ['16']),
    resolution=0.3,
    n_iterations=5
)

In [ ]:
# replace with your B cell clustering key
sc.tl.dendrogram(adata, groupby='leiden_R')

sc.pl.dotplot(
    adata,
    var_names=[
        # Stem & early progenitor markers
        'CD34', 'KIT', 'FLT3', 'HOXA9', 'MEIS1', 'LYL1', 'TCF3', 'GATA2',
        'IL7R', 'DNTT', 'RAG1', 'RAG2', 'PCNA', 'MKI67', 'TOP2A', 'STMN1',

        # Erythroid lineage
        'GYPA', 'HBA1', 'HBB', 'HBD', 'AHSP', 'ALAS2', 'KLF1', 'BCL11A', 'FECH', 'GATA1',

        # Megakaryocyte lineage
        'PF4', 'PPBP', 'ITGA2B', 'GP1BA', 'GP9', 'MPL', 'GATA1', 'MEIS1', 'CD34', 'CSF1R', 'CRYBG1',

        # Myeloid lineage
        'CEBPA', 'CEBPD', 'CSF1R', 'CSF3R', 'MPO', 'LYZ', 'IRF8', 'SPI1',

        # Dendritic cells / monocyte-DC precursors
        'BATF3', 'FCER1A', 'CLEC9A', 'CD1C',

        # B cell lineage
        'CD79A', 'CD79B', 'VPREB1', 'IGLL1', 'PAX5', 'MS4A1', 'MME',

        # T cell / lymphoid precursors
        'TCF3', 'IL7R', 'DNTT', 'RAG1', 'RAG2',

        # Stromal or supportive niche (likely contamination/background)
        'DCN', 'COL1A2', 'CXCL12', 'VCAN',

        # Others/unassigned
        'AVP', 'HES6'
    ],
    groupby='leiden_R',  # update to your relevant cluster key
    standard_scale='var',
    dendrogram=True,
    cmap='Reds',
    figsize=(22, 12),
    show=False
)

plt.savefig('figures/progs_dotplot.png', dpi=300, bbox_inches='tight')
plt.close()

In [ ]:
leiden_to_label = {'0': 'prog_b_late',
 '1': 'prog_b_early',
 '2': 'prog_b_proliferating',
 '3': 'prog_b_proliferating',
 '4,0': 'prog_hspc_cycling',
 '4,1': 'prog_b_proliferating',
 '5': 'prog_clp',
 '6': 'prog_lmpp',
 '7': 'mono_precursor_proliferating',
 '8': 'mono_precursor',
 '9': 'prog_cmp_granulocyte',
 '10,0': 'prog_hspc_multipotential',
 '10,1': 'prog_mep',
 '10,2': 'prog_mep',
 '10,3': 'prog_megakaryocyte',
 '11,0': 'prog_hspc_stem',
 '11,1': 'prog_hspc_multipotential',
 '11,2': 'prog_hspc_multipotential',
 '12': 'prog_mep',
 '13': 'junk_b',
 '14': 'prog_ery_mature',
 '15': 'prog_ery_pre',
 '16,0': 'junk_t',
 '16,1': 'junk_t',
 '16,2': 'junk_b',
 '16,3': 'dc_pdc',
 '16,4': 'junk_t',
 '17': 'prog_cmp',
 '18': 'prog_baeoma',
 '19': 'prog_ery_cycling',
 '20': 'prog_cmp',
 '21': 'junk_mono',
 '22': 'junk_t_prog'}
adata.obs['aifi_celltype_l3'] = adata.obs['leiden_R'].map(leiden_to_label).astype('category').cat.remove_unused_categories()

In [ ]:
adata.obs[['cell_uuid', 'aifi_celltype_l3']].to_parquet('../../../data/rna/bmmc-labels/progenitors_labels.parquet')

## 5. NK Cells Clean up

In [ ]:
adata = sc.read_h5ad('../../../data/rna/tmp/bmmc-annotation/nk_cells-processed-v1.h5ad')

In [ ]:
sc.tl.leiden(
    adata,
    restrict_to=('leiden', ['9']),
    resolution=0.8,
    n_iterations=2
)

sc.tl.leiden(
    adata,
    restrict_to=('leiden_R', ['12']),
    resolution=0.5,
    n_iterations=2
)

sc.tl.leiden(
    adata,
    restrict_to=('leiden_R', ['8']),
    resolution=0.2,
    n_iterations=2
)

In [ ]:
leiden_to_label = {'0': 'nk_effector',
 '1': 'nk_cd56_bright',
 '2': 'nk_effector',
 '3': 'nk_cd56_bright',
 '4': 'nk_effector',
 '5': 'nk_cd56_dim-gzmk_neg',
 '6': 'nk_cd56_dim-gzmk_pos',
 '7': 'nk_adaptive',
 '8,0': 'nk_cd56_dim-gzmk_pos',
 '8,1': 'nk_tissue_resident',
 '9,0': 'nk_t_proliferating_nk_like',
 '9,1': 'nk_t_proliferating_nk_like',
 '9,2': 'nk_t_proliferating_t_like',
 '9,3': 'nk_t_proliferating_nk_like',
 '9,4': 'nk_t_proliferating_nk_like',
 '9,5': 'nk_t_proliferating_nk_like',
 '9,6': 'nk_t_proliferating_t_like',
 '9,7': 'nk_t_proliferating_t_like',
 '9,8': 'nk_t_proliferating_nk_like',
 '9,9': 'nk_t_proliferating_t_like',
 '9,10': 'nk_t_proliferating_nk_like',
 '10': 'nk_cd56_dim-gzmk_neg',
 '11': 'nk_cd56_bright',
 '12,0': 'nk_cd56_bright',
 '12,1': 'nk_cd56_bright',
 '12,2': 'nk_cd56_bright',
 '12,3': 'nk_cd56_bright',
 '13': 'nk_cd56_dim-gzmk_pos',
 '14': 'nk_cd56_bright',
 '15': 'nk_cd56_dim-isg_pos',
 '16': 'nk_effector',
 '17': 'nk_cd56_bright'}

adata.obs['aifi_celltype_l3'] = adata.obs['leiden_R'].map(leiden_to_label).astype('category').cat.remove_unused_categories()

In [ ]:
adata.obs[['cell_uuid', 'aifi_celltype_l3']].to_parquet('../../../data/rna/bmmc-labels/nk_cells_labels.parquet')

## 6. CD 4 T cell Clean up

In [ ]:
adata = sc.read_h5ad('../../../data/rna/tmp/bmmc-annotation/t_cd4-processed-v1.h5ad')

In [ ]:
sc.tl.leiden(
    adata,
    restrict_to=('leiden', ['0']),
    resolution=0.8,
    n_iterations=2
)

sc.tl.leiden(
    adata,
    restrict_to=('leiden_R', ['1']),
    resolution=0.4,
    n_iterations=2
)

sc.tl.leiden(
    adata,
    restrict_to=('leiden_R', ['4']),
    resolution=0.4,
    n_iterations=2
)

sc.tl.leiden(
    adata,
    restrict_to=('leiden_R', ['7']),
    resolution=0.6,
    n_iterations=2
)

sc.tl.leiden(
    adata,
    restrict_to=('leiden_R', ['9']),
    resolution=0.4,
    n_iterations=2
)

sc.tl.leiden(
    adata,
    restrict_to=('leiden_R', ['12']),
    resolution=0.4,
    n_iterations=2
)

sc.tl.leiden(
    adata,
    restrict_to=('leiden_R', ['17']),
    resolution=0.4,
    n_iterations=2
)

In [ ]:
sc.tl.dendrogram(adata, groupby='leiden_R')
sc.pl.dotplot(
    adata,
    var_names=[
        # General
        'CD3E', 'CD3D', 'CD8A', 'CD4', 'MKI67',
        
        # CD4+ Naive and Central Memory
        'CCR7', 'SELL', 'IL7R', 'TCF7', 'LEF1',

        # CD4+ Effector and Memory
        'CD44', 'CXCR3', 'CCR6', 'IL2', 'TNF', 'IFNG', 'GZMA', 'GZMK', 'GZMB', 'PRF1',

        # CD4+ Regulatory T cells (Tregs)
        'FOXP3', 'IL2RA', 'IKZF2', 'CTLA4',

        # CD4+ ISG-high (interferon stimulated)
        'ISG15', 'IFI6', 'MX1', 'IFIT3',

        # CD8+ Naive and Effector
        'NKG7', 'CD3D',

        # Double Negative T cells (DN)
        'TRDC', 'KLRB1', 'ZBTB16',

        # ISG
        'MX1'
    ],
    groupby='leiden_R',
    standard_scale='var',
    dendrogram=True,
    cmap='Reds',
    figsize=(15, 12),
   show=False
)

plt.savefig('figures/t_cd4_dotplot.png', dpi=300, bbox_inches='tight')
plt.close()

In [ ]:
leiden_to_label = {'0,0': 't_cd4_central_memory',
 '0,1': 't_cd4_central_memory',
 '0,2': 't_cd4_effector_1',
 '0,3': 't_cd4_effector_1',
 '0,4': 't_cd4_central_memory',
 '0,5': 't_cd4_effector_1',
 '0,6': 't_cd4_effector_1',
 '0,7': 't_cd4_central_memory',
 '1,0': 't_cd4_central_memory',
 '1,1': 't_cd4_central_memory',
 '1,2': 't_cd4_naive',
 '2': 't_cd4_naive',
 '3': 't_cd4_naive',
 '4,0': 't_cd4_naive',
 '4,1': 't_cd4_central_memory',
 '5': 't_cd4_naive',
 '6': 't_cd4_naive',
 '7,0': 't_cd4_effector_2',
 '7,1': 't_cd4_effector_2',
 '7,2': 't_cd4_central_memory',
 '7,3': 't_cd4_central_memory',
 '8': 't_cd4_effector_1',
 '9,0': 't_cd4_effector_2',
 '9,1': 't_cd4_effector_2',
 '9,2': 't_cd4_memory',
 '9,3': 't_cd4_memory',
 '10': 't_cd4_central_memory',
 '11': 't_cd4_isg',
 '12,0': 't_cd4_central_memory',
 '12,1': 't_cd4_naive',
 '12,2': 't_cd4_naive',
 '13': 't_cd4_naive',
 '14': 't_cd8_naive',
 '15': 't_cd4_regs',
 '16': 't_cd4_regs',
 '17,0': 't_cd8_naive',
 '17,1': 't_cd8_naive',
 '17,2': 't_cd4_memory',
 '17,3': 't_cd8_naive',
 '17,4': 't_cd4_isg',
 '18': 't_cd4_naive',
 '19': 't_dn'}

adata.obs['aifi_celltype_l3'] = adata.obs['leiden_R'].map(leiden_to_label).astype('category').cat.remove_unused_categories()

In [ ]:
adata.obs[['cell_uuid', 'aifi_celltype_l3']].to_parquet('../../../data/rna/bmmc-labels/t_cd4_labels.parquet')

## 7. CD 8 T cell Clean up

In [2]:
adata = sc.read_h5ad('../../../data/rna/tmp/bmmc-annotation/t_cd8-processed-v1.h5ad')

In [3]:
sc.tl.leiden(
    adata,
    restrict_to=('leiden', ['8']),
    resolution=0.8,
    n_iterations=2
)

/tmp/ipykernel_343215/3663954065.py:1: FutureWarning: In the future, the default backend for leiden will be igraph instead of leidenalg.

 To achieve the future defaults please pass: flavor="igraph" and n_iterations=2.  directed must also be False to work with igraph's implementation.
  sc.tl.leiden(


In [4]:
sc.tl.dendrogram(adata, groupby="leiden_R")
sc.pl.dotplot(
    adata,
    var_names = [
    # General
    'CD3E', 'CD3D', 'CD8A', 'CD4', 'MKI67',
        
    # CD8+ Effector T cells
    "GZMB", "PRF1", "IFNG", "GNLY", "NKG7",

    # CD8+ Memory Tissue-Resident T cells
    "CD69", "ITGAE", "CXCR6", "ZEB2",

    # CD8+ Naive T cells
    "CCR7", "SELL", "IL7R", "LEF1", "TCF7",

    # Gamma Delta T cells (Tgd)
    "TRDC", "TRGC1", "TRGC2", "KLRB1", "CD3D",

    # MAIT cells
    "SLC4A10", "KLRB1", "TRAV1-2", "ZBTB16", "IL18RAP"],
    
    groupby='leiden_R',  
    standard_scale='var',
    dendrogram=True,
    cmap='Reds',
    figsize=(11, 12),
    show=False
)

plt.savefig("figures/t_cd8_dotplot.png", dpi=300, bbox_inches="tight")
plt.close()

In [5]:
leiden_to_label = {'0': 't_gd',
                   '1': 't_gd',
                   '2': 't_cd8_effector_1',
                   '3': 't_cd8_effector_1',
                   '4': 't_cd8_memory_tissue_resident',
                   '5': 't_cd8_effector_2',
                   '6': 't_cd8_effector_2',
                   '7': 't_cd8_effector_1',
                   '8,0': 't_cd8_effector_1',
                   '8,1': 't_cd8_naive',
                   '8,2': 't_cd8_effector_1',
                   '8,3': 't_cd8_naive',
                   '8,4': 't_cd8_naive',
                   '8,5': 't_cd8_naive',
                   '8,6': 't_cd8_naive',
                   '9': 't_cd8_effector_2',
                   '10': 't_cd8_effector_2',
                   '11': 't_cd8_effector_2',
                   '12': 't_cd8_memory_tissue_resident',
                   '13': 't_mait',
                   '14': 't_cd8_naive',
                   '15': 't_cd8_naive'}

adata.obs['aifi_celltype_l3'] = adata.obs['leiden_R'].map(leiden_to_label).astype('category').cat.remove_unused_categories()

In [6]:
adata.obs[['cell_uuid', 'aifi_celltype_l3']].to_parquet('../../../data/rna/bmmc-labels/t_cd8_labels.parquet')